In [1]:
import google.auth
import numpy as np
import pandas as pd
# import pygris 
# import pyogrio
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase
from shared_utils import arcgis_query

In [2]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [3]:
from calitp_data_analysis import get_fs
fs = get_fs()

In [4]:

import os
from typing import List, Optional, Union
import pyarrow.dataset as ds
from google.cloud import storage

In [5]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
from calitp_data_analysis.sql import to_snakecase

In [6]:
from typing import List

In [7]:
import geopandas as gpd
import pandas as pd

In [8]:
from functools import cache

from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [29]:
@cache
def gcs_pandas():
    return GCSPandas()

In [9]:
gcsgp = GCSGeoPandas()

# Census Blocks

In [10]:
census_year = 2020

In [11]:
geography_utils.CA_NAD83Albers_m 

'EPSG:3310'

In [12]:
census_gdf = to_snakecase(gcs_geopandas().read_parquet(f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_combined_{census_year}.parquet")).to_crs(geography_utils.CA_NAD83Albers_m )

## HPMS

In [13]:
def load_hpms(url:str)->gpd.GeoDataFrame:
    df = to_snakecase(gcsgp.read_parquet(
    hpms_url)).to_crs(geography_utils.CA_NAD83Albers_m )
    return df

In [14]:


hpms_url = "gs://calitp-analytics-data/data-analyses/equity_index/HPMS21_Main_SUCU.parquet"



In [15]:
hpm_df = load_hpms(hpms_url)

In [16]:
hpm_df.f_system.unique()

array([7, 5, 4, 3, 6, 2, 1], dtype=int32)

In [17]:
# Filter to f_system 1,2 temporarily
hpm_df2 = hpm_df.loc[hpm_df.f_system.isin([1,2])]

## Main function

In [18]:
buffer_list = [500, 450, 400, 350, 300, 250, 200, 150, 100, 50]

In [19]:
def buffer_intersect(hpm_gdf:gpd.GeoDataFrame, census_gdf:gpd.GeoDataFrame, buffer: int)->pd.DataFrame:
    hpm_gdf.geometry = hpm_gdf.geometry.buffer(buffer)
    
    intersect = gpd.overlay(hpm_gdf, census_gdf, how = "intersection",keep_geom_type= False)

    # Find area 
    intersect["area"] = intersect.geometry.area
    
    # Create an unique ID 
    intersect["unique_id"] = intersect.geoid20 + "_" + intersect.routeid

    # Find max value per unique ID
    intersect[f"aadt_max"] = intersect.groupby("unique_id")["aadt"].transform("max")

    # Sum the maximum AADT by GEOID
    agg = intersect.groupby("geoid20").agg({f"aadt_max":"sum", "area":"max"}).reset_index()

    # Calculate weighted aadt
    score_col_name = f"aadt_{buffer}_score"
    agg[score_col_name] = agg.aadt_max * agg.area
    agg2 = agg.groupby("geoid20").agg({score_col_name:"sum"}).reset_index()

    # Save to GCS
    agg2.to_parquet("./agg2.parquet")
    fs.put("./agg2.parquet", f"calitp-analytics-data/data-analyses/equity_index/eqi_traffic_{buffer}.parquet")
    print(f"finshed saving data for {buffer}")
    return agg2

In [20]:
# buffer_500 = buffer_intersect(hpm_gdf=hpm_df2, census_gdf=census_gdf, buffer = buffer_list[0])

In [21]:
#for buffer in [200, 150, 100, 50]:
#    df = buffer_intersect(hpm_gdf=hpm_df2, census_gdf=census_gdf, buffer = buffer)

In [32]:
df = gcs_pandas().read_parquet("gs://calitp-analytics-data/data-analyses/equity_index/eqi_traffic_100.parquet")

In [33]:
df.sample()

,geoid20,aadt_100_score
101199,061130112043011,3232676943.56


In [43]:

def load_eqi_traffic_parquets():
    """
    Reads all EQI traffic parquet files from the GCS folder
    and concatenates them together on the column 'geoid20'.
    """

    base_path = "gs://calitp-analytics-data/data-analyses/equity_index/eqi_traffic_"
    sizes = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]

    dfs = []

    for size in sizes:
        path = f"{base_path}{size}.parquet"
        df = gcs_pandas().read_parquet(path)
        dfs.append(df)

   # Merge all dfs on geoid20
    combined = dfs[0]
    for df in dfs[1:]:
        combined = combined.merge(df, on="geoid20", how="outer")


    # Optional: ensure geoid20 exists
    if "geoid20" not in combined.columns:
        raise ValueError("Column 'geoid20' not found in concatenated data.")

    return combined


In [53]:
df = load_eqi_traffic_parquets()

In [54]:
df.head(3)

,geoid20,aadt_50_score,aadt_100_score,aadt_150_score,aadt_200_score,aadt_250_score,aadt_300_score,aadt_350_score,aadt_400_score,aadt_450_score,aadt_500_score
0,060014001001010,NaN,NaN,NaN,NaN,6586657633.82,NaN,NaN,2567185696.38,NaN,23090352783.57
1,060014001001011,NaN,NaN,NaN,NaN,37910564373.46,NaN,NaN,21896357554.63,NaN,122354533713.92
2,060014001001012,NaN,NaN,NaN,NaN,8841364537.69,NaN,NaN,4555636435.20,NaN,27213652351.68


In [56]:
df.columns

Index(['geoid20', 'aadt_50_score', 'aadt_100_score', 'aadt_150_score',
       'aadt_200_score', 'aadt_250_score', 'aadt_300_score', 'aadt_350_score',
       'aadt_400_score', 'aadt_450_score', 'aadt_500_score', 'p_50_m'],
      dtype='object')

In [55]:
df["p_50_m"]  = df["aadt_50_score"]

In [57]:
df["p_100_m"] = df["aadt_100_score"] - df["aadt_50_score"]

In [58]:
df["p_150_m"] = df["aadt_150_score"] - df["aadt_100_score"]

In [59]:
df["p_200_m"] = df["aadt_200_score"] - df["aadt_150_score"]

In [60]:
df["p_250_m"] = df["aadt_250_score"] - df["aadt_200_score"]

In [61]:
df["p_300_m"] = df["aadt_300_score"] - df["aadt_250_score"]

In [67]:
df["p_350_m"] = df["aadt_350_score"] - df["aadt_250_score"]

In [66]:
df["p_400_m"] = df["aadt_400_score"] - df["aadt_350_score"]

In [65]:
df["p_450_m"] = df["aadt_450_score"] - df["aadt_400_score"]

In [70]:
df["p_500_m"] = df["aadt_500_score"] - df["aadt_450_score"]

In [68]:
weights = {
	        "p_50_m": 1.0,
	        "p_100_m": 0.5,
	        "p_150_m": 0.33,
	        "p_200_m": 0.25,
	        "p_250_m": 0.20,
	        "p_300_m": 0.16,
	        "p_350_m": 0.14,
	        "p_400_m": 0.125,
	        "p_450_m": 0.111,
	        "p_500_m": 0.10,
    }

In [71]:
for col, w in weights.items():
    df[f"{col}_w"] = df[col] * w

In [72]:
weighted_cols = [f"{col}_w" for col in weights.keys()]
df["weighted_aadt_score"] = df[weighted_cols].sum(axis=1)

In [73]:
df.head(2)

,geoid20,aadt_50_score,aadt_100_score,aadt_150_score,aadt_200_score,aadt_250_score,aadt_300_score,aadt_350_score,aadt_400_score,aadt_450_score,aadt_500_score,p_50_m,p_100_m,p_150_m,p_200_m,p_250_m,p_300_m,p_350_m,p_400_m,p_450_m,p_50_m_w,p_100_m_w,p_150_m_w,p_200_m_w,p_250_m_w,p_300_m_w,p_350_m_w,p_400_m_w,p_450_m_w,p_500_m,p_500_m_w,weighted_aadt_score
0,060014001001010,NaN,NaN,NaN,NaN,6586657633.82,NaN,NaN,2567185696.38,NaN,23090352783.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
1,060014001001011,NaN,NaN,NaN,NaN,37910564373.46,NaN,NaN,21896357554.63,NaN,122354533713.92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00


In [75]:
df2 = df[["geoid20", "weighted_aadt_score"]].copy().fillna(0)

In [76]:
df2["traffic_proximity_and_volume_percentile"] = (
	        df2["weighted_aadt_score"].rank(pct=True)
    )

In [77]:
df2.head()

,geoid20,weighted_aadt_score,traffic_proximity_and_volume_percentile
0,060014001001010,0.00,0.33
1,060014001001011,0.00,0.33
2,060014001001012,0.00,0.33
3,060014001001018,0.00,0.33
4,060014001001019,0.00,0.33


In [78]:
df2.weighted_aadt_score.describe()

count           168161.00
mean       43741792251.95
std       138065973207.88
min      -226913168178.24
25%                  0.00
50%         3226140493.38
75%        39126485904.70
max     10118644533307.61
Name: weighted_aadt_score, dtype: float64

In [79]:
df2.traffic_proximity_and_volume_percentile.describe()

count   168161.00
mean         0.50
std          0.29
min          0.00
25%          0.33
50%          0.50
75%          0.75
max          1.00
Name: traffic_proximity_and_volume_percentile, dtype: float64

In [88]:
census_gdf.columns

Index(['statefp20', 'countyfp20', 'tractce20', 'blockce20', 'geoid20',
       'name20', 'mtfcc20', 'ur20', 'uace20', 'uatype20', 'funcstat20',
       'aland20', 'awater20', 'intptlat20', 'intptlon20', 'housing20', 'pop20',
       'b250', 'county_name'],
      dtype='object')

In [89]:
type(census_gdf)

geopandas.geodataframe.GeoDataFrame

In [91]:
m1 = pd.merge(census_gdf[["geoid20","county_name", "b250"]], df2, on = ["geoid20"], how = "left")

In [92]:
m1.shape

(519723, 5)

In [95]:
m1.columns

Index(['geoid20', 'county_name', 'b250', 'weighted_aadt_score',
       'traffic_proximity_and_volume_percentile'],
      dtype='object')

ERROR! Session/line number was not unique in database. History logging moved to new session 27


In [94]:
m1.county_name.unique()

array(['Alameda', 'Alpine', 'Amador', 'Butte', 'Calaveras', 'Colusa',
       'Contra Costa', 'Del Norte', 'El Dorado', 'Fresno', 'Glenn',
       'Humboldt', 'Imperial', 'Inyo', 'Kern', 'Kings', 'Lake', 'Lassen',
       'Los Angeles', 'Madera', 'Marin', 'Mariposa', 'Mendocino',
       'Merced', 'Modoc', 'Mono', 'Monterey', 'Napa', 'Nevada', 'Orange',
       'Placer', 'Plumas', 'Riverside', 'Sacramento', 'San Benito',
       'San Bernardino', 'San Diego', 'San Francisco', 'San Joaquin',
       'San Luis Obispo', 'San Mateo', 'Santa Barbara', 'Santa Clara',
       'Santa Cruz', 'Shasta', 'Sierra', 'Siskiyou', 'Solano', 'Sonoma',
       'Stanislaus', 'Sutter', 'Tehama', 'Trinity', 'Tulare', 'Tuolumne',
       'Ventura', 'Yolo', 'Yuba'], dtype=object)

In [ ]:
m

In [97]:
m1.loc[(m1.county_name == "San Francisco")&(m1.traffic_proximity_and_volume_percentile > 0.9)].explore("traffic_proximity_and_volume_percentile")